# 02 Exploratory Data Analysis

This notebook explores the cleaned 2025 Stack Overflow Developer Survey data prepared in `01_data_check.ipynb`.

The goal is to understand patterns in:
- job satisfaction
- career change consideration
- learning pathways
- compensation

Learning pathway flags are **not mutually exclusive**. Respondents may select multiple learning methods, so pathway charts should be interpreted descriptively rather than as separate independent groups.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../data/processed/stackoverflow_2025_project_cleaned.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

## Job Satisfaction Distribution

`JobSat` is a 0–10 scale measuring how satisfied respondents are in their current professional developer role.

In [ ]:
jobsat_counts = df["JobSat"].value_counts(dropna=False).sort_index()
jobsat_counts

In [ ]:
jobsat_counts_no_missing = df["JobSat"].dropna().value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(jobsat_counts_no_missing.index, jobsat_counts_no_missing.values)
plt.xlabel("Job Satisfaction Score")
plt.ylabel("Number of Respondents")
plt.title("Distribution of Job Satisfaction Scores")
plt.xticks(range(0, 11))
plt.show()

In [ ]:
df["JobSat"].describe()

## Career Change Consideration

The `NewRole` column asks whether respondents considered a career change or transitioned into a new career or industry in the past year. This project converts that response into a binary variable called `career_change_considered`.

This should **not** be described as attrition or retention. It is a career-change consideration/transition signal.

In [ ]:
df["NewRole"].value_counts(dropna=False)

In [ ]:
career_counts = df["career_change_considered"].value_counts(dropna=False)
career_counts

In [ ]:
career_labels = {
    0.0: "No career change considered",
    1.0: "Considered or transitioned"
}

career_counts_labeled = (
    df["career_change_considered"]
    .map(career_labels)
    .value_counts(dropna=False)
)

career_counts_labeled

In [ ]:
career_percent = (
    df["career_change_considered"]
    .map(career_labels)
    .value_counts(normalize=True, dropna=True)
    * 100
).round(1)

career_percent

In [ ]:
career_counts_plot = (
    df["career_change_considered"]
    .map(career_labels)
    .value_counts()
)

plt.figure(figsize=(7, 5))
plt.bar(career_counts_plot.index, career_counts_plot.values)
plt.xlabel("Career Change Group")
plt.ylabel("Number of Respondents")
plt.title("Career Change Consideration")
plt.xticks(rotation=15, ha="right")
plt.show()

Among respondents who answered the `NewRole` question, slightly more than half reported that they had either considered a career or industry change or had transitioned into a new career or industry in the past year. This makes the variable useful for classification because the target is relatively balanced rather than being dominated by one class.

## Job Satisfaction and Career Change Consideration

This section compares job satisfaction between respondents who did and did not consider or transition into a new career or industry.

In [ ]:
jobsat_by_career_table = (
    df.dropna(subset=["career_change_considered", "JobSat"])
    .assign(career_group=lambda x: x["career_change_considered"].map(career_labels))
    .groupby("career_group")["JobSat"]
    .agg(["count", "mean", "median", "std"])
)

jobsat_by_career_table

In [ ]:
jobsat_by_career = jobsat_by_career_table["mean"].sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(jobsat_by_career.index, jobsat_by_career.values)
plt.xlabel("Career Change Group")
plt.ylabel("Average Job Satisfaction")
plt.title("Average Job Satisfaction by Career Change Consideration")
plt.ylim(0, 10)
plt.xticks(rotation=15, ha="right")
plt.show()

Respondents who did not consider or transition into a new career or industry reported higher average job satisfaction than those who did. This supports using job satisfaction as a meaningful workforce outcome in the project.

## Learning Pathway EDA

This section explores how respondents learned to code. These pathway flags are not mutually exclusive. A respondent may appear in multiple categories.

In [ ]:
learning_pathway_flags = {
    "learned_online_courses": "Online Courses or Certification",
    "learned_school": "School",
    "learned_bootcamp": "Coding Bootcamp",
    "learned_ai": "AI CodeGen",
    "learned_docs": "Technical documentation",
    "learned_on_job": "Colleague",
    "learned_books": "Books",
    "learned_videos": "Videos",
}

pretty_learning_labels = {
    "learned_online_courses": "Online courses/certification",
    "learned_school": "School/university",
    "learned_bootcamp": "Coding bootcamp",
    "learned_ai": "AI tools",
    "learned_docs": "Technical documentation",
    "learned_on_job": "On-the-job/colleague",
    "learned_books": "Books/physical media",
    "learned_videos": "Videos",
}

# Recreate the pathway flags if they are missing from the saved file.
# This makes the EDA notebook more robust after a kernel restart.
for flag_col, label in learning_pathway_flags.items():
    if flag_col not in df.columns:
        df[flag_col] = df["LearnCode"].str.contains(label, na=False).astype(int)

learning_flags = list(learning_pathway_flags.keys())

### Learning Pathway Counts

This table shows how often each learning pathway appears in the survey responses.

In [ ]:
learning_counts = df[learning_flags].sum().sort_values(ascending=False)
learning_counts_pretty = learning_counts.rename(index=pretty_learning_labels)
learning_counts_pretty

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(learning_counts_pretty.index, learning_counts_pretty.values)
plt.xlabel("Number of Respondents")
plt.ylabel("Learning Pathway")
plt.title("Most Common Learning Pathways")
plt.gca().invert_yaxis()
plt.show()

Technical documentation, videos, AI tools, and online courses/certifications appear more often than formal school or coding bootcamp pathways. Because respondents could select more than one pathway, these counts should be interpreted as descriptive learning signals rather than mutually exclusive groups.

### Job Satisfaction by Learning Pathway

In [ ]:
jobsat_by_pathway = {}

for col in learning_flags:
    jobsat_by_pathway[col] = df.loc[df[col] == 1, "JobSat"].mean()

jobsat_by_pathway = pd.Series(jobsat_by_pathway).sort_values(ascending=False)
jobsat_by_pathway_pretty = jobsat_by_pathway.rename(index=pretty_learning_labels)

jobsat_by_pathway_pretty

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(jobsat_by_pathway_pretty.index, jobsat_by_pathway_pretty.values)
plt.xlabel("Average Job Satisfaction")
plt.ylabel("Learning Pathway")
plt.title("Average Job Satisfaction by Learning Pathway")
plt.xlim(0, 10)
plt.gca().invert_yaxis()
plt.show()

Average job satisfaction differs slightly across learning pathways, but these comparisons are descriptive only. Later modeling will account for other factors such as experience, compensation, education, and workplace characteristics.

### Career Change Consideration by Learning Pathway

In [ ]:
career_by_pathway = {}

for col in learning_flags:
    career_by_pathway[col] = df.loc[df[col] == 1, "career_change_considered"].mean() * 100

career_by_pathway = pd.Series(career_by_pathway).sort_values(ascending=False)
career_by_pathway_pretty = career_by_pathway.rename(index=pretty_learning_labels)

career_by_pathway_pretty

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(career_by_pathway_pretty.index, career_by_pathway_pretty.values)
plt.xlabel("Percent Considered or Transitioned")
plt.ylabel("Learning Pathway")
plt.title("Career Change Consideration by Learning Pathway")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

## Compensation Sanity Check

Compensation contains major outliers, so the project keeps the original annual compensation column for EDA and uses `log_salary` as a better-behaved modeling feature.

In [ ]:
df["ConvertedCompYearly"].describe()

In [ ]:
df["log_salary"].describe()

In [ ]:
salary_under_300k = df[df["ConvertedCompYearly"] <= 300000]["ConvertedCompYearly"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(salary_under_300k, bins=40)
plt.xlabel("Annual Compensation")
plt.ylabel("Number of Respondents")
plt.title("Distribution of Annual Compensation Under $300,000")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["log_salary"].dropna(), bins=40)
plt.xlabel("Log Salary")
plt.ylabel("Number of Respondents")
plt.title("Distribution of Log-Transformed Salary")
plt.show()

## EDA Summary

Initial exploratory analysis showed that the dataset supports the planned capstone.

Key observations:
- Job satisfaction is concentrated toward the higher end of the 0–10 scale.
- The career change consideration target is relatively balanced among respondents who answered the question.
- Respondents who did not consider or transition into a new career or industry reported higher average job satisfaction.
- Learning pathways vary widely, with technical documentation, videos, AI tools, and online courses/certifications appearing most often.
- Learning pathway groups overlap, so pathway charts are descriptive rather than causal.
- Compensation has large outliers, so log-transformed salary is more appropriate for modeling than raw annual compensation.